In [31]:
from mini_competition.data_functions import load_data, add_date_features, clean_features, add_zero_sales, PROCCESED_PATH
import pandas as pd

data = load_data(PROCCESED_PATH / "train.parquet" )
data.info

<bound method DataFrame.info of                 date  store_nbr  item_nbr  unit_sales onpromotion
0         2013-01-01         25    103665         7.0        None
1         2013-01-01         25    105574         1.0        None
2         2013-01-01         25    105575         2.0        None
3         2013-01-01         25    108079         1.0        None
4         2013-01-01         25    108701         1.0        None
...              ...        ...       ...         ...         ...
125497035 2017-08-15         54   2089339         4.0       False
125497036 2017-08-15         54   2106464         1.0        True
125497037 2017-08-15         54   2110456       192.0       False
125497038 2017-08-15         54   2113914       198.0        True
125497039 2017-08-15         54   2116416         2.0       False

[125497040 rows x 5 columns]>

In [ ]:
data = add_zero_sales(data)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import TargetEncoder, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np

def clean_features(d):
    d = d.copy()
    d["onpromotion"] = d["onpromotion"].fillna(False)
    return d

def add_date_features(d):
    d = d.copy()
    # adds weekday and month feature
    d["date"] = pd.to_datetime(d["date"])
    d["weekday_num"] = d["date"].dt.day_of_week
    d["month_num"] = d["date"].dt.month
    return d.drop(columns="date")


def add_lag_features(df : pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)
    df["log_sales"] = np.log1p(df["unit_sales"].astype("float32"))

    g = df.groupby(["store_nbr", "item_nbr"])["log_sales"]

    for lag in [16, 21, 28, 35]:
        df[f"lag_{lag}"] = g.shift(lag)

    df["rolling_mean_7"]  = g.transform(lambda s: s.shift(16).rolling(7).mean())
    df["rolling_mean_28"] = g.transform(lambda s: s.shift(16).rolling(28).mean())

    return df.copy()

def drop_lag_none_rows(df : pd.DataFrame) -> pd.DataFrame:
    cols = ["lag_16", "lag_21","lag_28", "lag_35","rolling_mean_7", "rolling_mean_28"]
    df = df.dropna(subset=cols, how="any")

df = drop_lag_none_rows(add_lag_features(data))

train = df[(df["date"] < "2017-07-31") & (df["date"] > "2016-07-31")].sample(n=2_000_000, random_state=0)
valid = df[df["date"] >= "2017-07-31"].sample(n=400_000, random_state=0)

# Use the raw item_nbr and the raw unit_sales; the pipeline handles both
features = ["date", "store_nbr", "item_nbr", "onpromotion"]
X_train, y_train = train[features], train["unit_sales"]
X_valid, y_valid = valid[features], valid["unit_sales"]

# Target-encode item_nbr and pass the other columns through unchanged
preprocess = ColumnTransformer(
    [("item_te", TargetEncoder(), ["item_nbr"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")   # keeps store_nbr as 'category'

model = TransformedTargetRegressor(
    regressor=make_pipeline(
        FunctionTransformer(clean_features),
        FunctionTransformer(add_date_features),
        preprocess,
        HistGradientBoostingRegressor(
            categorical_features=["store_nbr"],
            max_iter=1000, learning_rate=0.05, max_leaf_nodes=127,
            early_stopping=True,
        ),
    ),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,   # clipping can't be undone, so skip the round-trip check
)


model.fit(X_train, y_train)
pred = model.predict(X_valid)